In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import io

In [4]:
# 1. КОНСТАНТЫ И КЛАССЫ (те же самые 33 класса)
CLASS_NAMES = [
    'Apple - Apple scab', 'Apple - Black rot', 'Apple - Cedar apple rust', 'Apple - healthy',
    'Cherry - Powdery mildew', 'Cherry - healthy',
    'Corn - Cercospora leaf spot', 'Corn - Common rust', 'Corn - Northern Leaf Blight', 'Corn - healthy',
    'Grape - Black rot', 'Grape - Esca (Black Measles)', 'Grape - Leaf blight', 'Grape - healthy',
    'Peach - Bacterial spot', 'Peach - healthy',
    'Pepper bell - Bacterial spot', 'Pepper bell - healthy',
    'Potato - Early blight', 'Potato - Late blight', 'Potato - healthy',
    'Strawberry - Leaf scorch', 'Strawberry - healthy',
    'Tomato - Bacterial spot', 'Tomato - Early blight', 'Tomato - Late blight',
    'Tomato - Leaf Mold', 'Tomato - Septoria leaf spot', 'Tomato - Spider mites',
    'Tomato - Target Spot', 'Tomato - Yellow Leaf Curl Virus', 'Tomato - mosaic virus', 'Tomato - healthy'
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [5]:
# 2. ИНИЦИАЛИЗАЦИЯ МОДЕЛИ
# Создаем структуру EfficientNet-B0
model_eval = models.efficientnet_b0(weights=None)
model_eval.classifier[1] = nn.Linear(model_eval.classifier[1].in_features, 33)

# ЗАГРУЗКА ВЕСОВ (убедись, что загрузил файл .pth в файлы Colab слева)
model_path = 'best_plant_disease_cv_model_clean.pth'
model_eval.load_state_dict(torch.load(model_path, map_location=device))
model_eval.to(device)
model_eval.eval()


FileNotFoundError: [Errno 2] No such file or directory: 'best_plant_disease_cv_model_clean.pth'

In [ ]:
# 3. ПОДГОТОВКА ТРАНСФОРМАЦИЙ
test_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


In [ ]:
# 4. ЗАГРУЗКА ФАЙЛА И ПРЕДИКТ
print("Нажми на кнопку ниже, чтобы выбрать фото листа:")
uploaded = files.upload()

for filename in uploaded.keys():
    # Открываем изображение
    image_data = uploaded[filename]
    img = Image.open(io.BytesIO(image_data)).convert('RGB')

    # Визуализация загруженной картинки
    img.show()

    # Обработка для нейросети
    tensor = test_transforms(img).unsqueeze(0).to(device)

    # Предсказание
    with torch.no_grad():
        outputs = model_eval(tensor)
        probabilities = F.softmax(outputs, dim=1)
        confidence, predicted_idx = torch.max(probabilities, 1)

    result = CLASS_NAMES[predicted_idx.item()]
    conf_score = confidence.item() * 100

    print(f"\n--- РЕЗУЛЬТАТ АНАЛИЗА ---")
    print(f"Файл: {filename}")
    print(f"Диагноз: {result}")
    print(f"Уверенность: {conf_score:.2f}%")

    if "healthy" in result.lower():
        print("✅ Растение здорово!")
    else:
        print("⚠️ Обнаружено заболевание. Требуется лечение.")